In [2]:
# Ensure the notebook has required packages and is running from the project root
import os
import sys
from pathlib import Path
import subprocess

p = Path().resolve()
project_root = None
for parent in [p] + list(p.parents):
    if (parent / "config" / "config.yaml").exists():
        project_root = parent
        break
if project_root:
    os.chdir(str(project_root))
    print("Set notebook cwd to project root:", project_root)
else:
    print("Project root with config/config.yaml not found; cwd unchanged:", p)

required_packages = [
    "accelerate>=0.26.0",
    "absl-py",
    "rouge_score",
    "nltk"
]
for pkg in required_packages:
    try:
        __import__(pkg.split("==")[0].split(">=")[0].split("<")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
print("Required packages installed.")


Set notebook cwd to project root: C:\Users\banot\text-summarizer-


c:\Users\banot\anaconda3\envs\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Required packages installed.


In [15]:
import sys
import accelerate
import transformers
print('python executable:', sys.executable)
print('accelerate version:', accelerate.__version__)
print('transformers version:', transformers.__version__)


python executable: c:\Users\banot\anaconda3\envs\textS\python.exe
accelerate version: 1.0.1
transformers version: 4.46.3


In [1]:
import os

In [2]:
%pwd

'c:\\Users\\banot\\text-summarizer-\\research'

In [3]:
os.chdir("../")

In [4]:
try:
    import hf_xet
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hf_xet"])
    import hf_xet

print("hf_xet installed and ready")

hf_xet installed and ready


In [5]:
%pwd

'c:\\Users\\banot\\text-summarizer-'

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [5]:

from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )

        return model_trainer_config

In [7]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
from huggingface_hub import snapshot_download
import torch
import time

[2026-07-30 12:03:34,512: INFO: config: PyTorch version 2.4.1 available.]


In [8]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        model_ckpt = str(self.config.model_ckpt)
        if not Path(model_ckpt).exists():
            download_attempts = 3
            last_exception = None
            for attempt in range(1, download_attempts + 1):
                try:
                    model_ckpt = snapshot_download(
                        repo_id=model_ckpt,
                        resume_download=True,
                        max_workers=1,
                    )
                    break
                except Exception as e:
                    last_exception = e
                    if attempt < download_attempts:
                        time.sleep(5)
                    else:
                        raise RuntimeError(
                            f"Failed to download Hugging Face checkpoint '{self.config.model_ckpt}'. "
                            "Check your network and retry."
                        ) from e

        tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
        
        # loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            no_cuda=True,
        ) 

        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))


In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-07-30 12:03:41,026: INFO: common: YAML file loaded successfully: config\config.yaml]
[2026-07-30 12:03:41,032: INFO: common: YAML file loaded successfully: params.yaml]
[2026-07-30 12:03:41,033: INFO: common: Created directory at: artifacts]
[2026-07-30 12:03:41,034: INFO: common: Created directory at: artifacts/model_trainer]


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]c:\Users\banot\anaconda3\envs\textS\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 703.01it/s]
Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at C:\Users\banot\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail\snapshots\40d588fdab0cc077b80d950b300bf66ad3c75b92 and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\banot\anaconda3\envs\textS\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecate

{'loss': 3.1995, 'grad_norm': 322.7123107910156, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.01}


  2%|▏         | 20/920 [41:38<28:46:56, 115.13s/it]

{'loss': 3.2262, 'grad_norm': 156.32550048828125, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.02}


...

  7%|▋         | 60/920 [1:28:47<16:28:54, 68.99s/it]

{'loss': 2.7268, 'grad_norm': 203.43507385253906, 'learning_rate': 6e-06, 'epoch': 0.07}


  8%|▊         | 70/920 [1:37:46<11:42:04, 49.56s/it]

{'loss': 2.6268, 'grad_norm': 165.99314880371094, 'learning_rate': 7.000000000000001e-06, 'epoch': 0.08}


  8%|▊         | 75/920 [1:44:01<15:57:04, 67.96s/it]

In [1]:
from transformers import pipeline

summarizer = pipeline("summarization", model="t5-small")

text = """
Artificial Intelligence is transforming industries by automating processes,
improving efficiency, and enabling data-driven decisions.
"""

summary = summarizer(text, max_length=50, min_length=10, do_sample=False)

print(summary[0]['summary_text'])

c:\Users\banot\anaconda3\envs\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Your max_length is set to 50, but your input_length is only 27. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=13)


Artificial Intelligence is transforming industries by automating processes, improving efficiency, and enabling data-driven decisions.
